# Lesson 4 : Hosted Tools

Agent Framework provides a wide variety of built-in tool's object (such as, Code Interpreter, Web Search, File Search, MCP tools, etc), and you can use these useful tools in your agent.<br>
With ```AzureAIClient``` in Agent Framwork, there exist 3 types of tools as follows. :

- Tools hosted natively in Foundry : As I have mentioned in Lesson 1, ```AzureAIClient``` uses Azure AI Projects SDK (```azure-ai-projects```) v2. You can work with tools natively supported in Azure AI Projects SDK - i.e., in Microsoft Foundry. The available built-in tools will vary depending on the type of client. For example, you can use [Claude's web search tool](https://platform.claude.com/docs/en/agents-and-tools/tool-use/web-search-tool) when ```AnthropicClient```.
- Tools in Foundry catalog : Microsoft Foundry provides various additional tools in the gallery (catalog) - such as, SharePoint tool, Fabric data agent tool, OpenAPI-integrated tool, or 3rd-party tools. We'll see how to use these tools in the next Lesson 5.
- Tools in Agent Framework : The library of Agent Framework SDK also provides native tools. Unlike above tool callings (server-side tool calling), these tools are mostly handled as local functions in LLM invocation, and these are processed in Agent Framework SDK which runs in your local computers (i.e., client-side tool calling). For example, native MCP tools in Agent Framework (such as, ```MCPStdioTool```, ```MCPStreamableHTTPTool```, or ```MCPWebsocketTool```) are locally processed in Agent Framework along with MCP protocol specification.

In this example, we will explore web search tool and MCP tool, which are natively hosted in Microsoft Foundry (i.e., above 1st case).

---

Important Note : Currently, there is a bug in ```AzureAIClient``` that causes a warning when :
- You set ```use_latest_version=True``` option and get an agent.
- Run agent with the same ```AzureAIClient``` more than once.

Please ignore warning.

---

## Web Search tool

Same as in Lesson 1, we create a ```AzureAIClient``` object as follows.

In [1]:
from dotenv import load_dotenv
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

load_dotenv()

credential = AzureCliCredential()
client = AzureAIClient(
    credential=credential,
)

Now we create tool definition for native web search tool in Microsoft Foundry, and create an agent with this tool setting.

By running ```get_web_search_tool()``` in ```AzureAIClient```, ```WebSearchPreviewTool``` object is internally created and returned. (Using Azure AI Projects SDK, you can also explicitly create ```WebSearchPreviewTool``` object by yourself.)

In [2]:
from agent_framework import Agent

web_search_tool = client.get_web_search_tool()
agent = Agent(
    name="WeatherAgentWithSearchTool",
    client=client,
    instructions="You are an agent about weather information.",
    tools=[web_search_tool])

Now we run the agent as follows.

As you see, this agent knows "what day is it today" or "what is the actual weather condition today", because web search tool is used internally.

In Lesson 1, the tool execution is run on your local function. In this example, Microsoft Foundry will handle this process on server.

In [3]:
from IPython.display import Markdown, display

result = await agent.run("Tell me the weather and temperature in Osaka today.")
display(Markdown(result.text))

Today in **Osaka (Fri, Jan 16, 2026)**: **mostly sunny**.

- **Temperature:** around **15°C / 59°F** high, **~5°C / 41°F** low [Osaka-shi, Osaka, Japan Weather Forecast | AccuWeather](https://www.accuweather.com/en/jp/osaka-shi/225007/weather-forecast/225007)[10 Day Weather - Osaka, Osaka, Japan - The Weather Channel](https://weather.com/weather/tenday/l/Osaka+Osaka+Japan?placeId=441174f51a1951566e6b1d02bd724effab80d7359e5f241540d1aa46dfecc59f)[Osaka, Japan 14 day weather forecast - timeanddate.com](https://www.timeanddate.com/weather/japan/osaka/ext)[Weather - Osaka City - 14-Day Forecast & Rain | Ventusky](https://www.ventusky.com/osaka)[Osaka Weather Forecast](https://www.weather-forecast.com/locations/Osaka/forecasts/latest)

## MCP tool

Next we explore native MCP tool in Microsoft Foundry with Agent Framework. (This will internally use MCP tool definition in Azure OpenAI Responses API.)

Same as above example, ```get_mcp_tool()``` method in ```AzureAIClient``` creates and returns ```MCPTool``` object.

> Note : As I have mentioned above, Agent Framework also provides native built-in MCP tools - such as, ```MCPStdioTool```, ```MCPStreamableHTTPTool```, and ```MCPWebsocketTool```. These are processed as local functions in Agent Framework SDK, in accordance with the MCP protocol specifications.  
> These might be useful when MCP (or some part of MCP specification) is not supported in your client. (For example, some remote API models might not support MCP Stdio server's tool, but ```MCPStdioTool``` can do.)

In this example, we create an agent to answer Microsoft technical questions.  
This agent uses a remote MCP server (Streamable HTTP server), which provides information about Microsoft Learn document.

In [4]:
mcp_tool = client.get_mcp_tool(
    name="Microsoft Learn MCP",
    url="https://learn.microsoft.com/api/mcp",
    approval_mode="never_require",
)
agent = Agent(
    name="MSTechKnowledgeAgent",
    client=client,
    instructions="You are an agent who answers technical questions about Microsoft products and services.",
    tools=[mcp_tool],
)

Let's ask a technical question about Microsoft Azure.  
In this call, MCP tool calling (about Microsoft Learn document) is handled in Microsoft Foundry. (Use [tracing](./02_trace.ipynb) and see the internal steps.)

In [5]:
from IPython.display import Markdown, display

result = await agent.run("How to create an Azure storage account using Azure CLI ?")
display(Markdown(result.text))

To create an Azure Storage account with the Azure CLI, you typically:

1) **Sign in and pick a subscription**
```bash
az login
az account set --subscription "<SUBSCRIPTION_ID_OR_NAME>"
```

2) **Create a resource group** (skip if you already have one)
```bash
az group create \
  --name <RG_NAME> \
  --location <LOCATION>
```

3) **Create the storage account**
```bash
az storage account create \
  --name <STORAGE_ACCOUNT_NAME> \
  --resource-group <RG_NAME> \
  --location <LOCATION> \
  --sku Standard_LRS \
  --kind StorageV2
```

### Common options you may want
- **Block public blob access** (recommended in many orgs):
```bash
az storage account create \
  --name <STORAGE_ACCOUNT_NAME> \
  --resource-group <RG_NAME> \
  --location <LOCATION> \
  --sku Standard_LRS \
  --kind StorageV2 \
  --allow-blob-public-access false
```

- **Require secure transfer (HTTPS only)**:
```bash
az storage account update \
  --name <STORAGE_ACCOUNT_NAME> \
  --resource-group <RG_NAME> \
  --https-only true
```

4) **Verify**
```bash
az storage account show \
  --name <STORAGE_ACCOUNT_NAME> \
  --resource-group <RG_NAME>
```

### Notes / gotchas
- `<STORAGE_ACCOUNT_NAME>` must be **globally unique**, **3–24** chars, **lowercase letters and numbers only**.
- `<LOCATION>` examples: `eastus`, `westus2`, `westeurope`.

If you tell me your intended region, redundancy (LRS/ZRS/GRS), and whether this is for blobs only or general use, I can suggest the best `--sku`/security flags.